[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-username/tensor-forge/blob/main/fable_folder/notebooks/12_art_of_action.ipynb)

# ⚒️ Act III · Quest 12 — The Art of Action

*No labels, no targets — just consequences. Reinforcement learning on CartPole.*

⬅️ [11_art_of_creation](11_art_of_creation.ipynb)  •  [13_art_of_speed](13_art_of_speed.ipynb) ➡️

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math, random

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')
np.random.seed(0); random.seed(0)

# ══════════════════ ⚒️ FORGE GRADER — run me once ══════════════════
# Powers check() and xp_report(). Exercises give instant feedback + XP.
_XP = {"earned": 0, "done": set(), "checks": {}}

def _register(name, points, test, hint):
    _XP["checks"][name] = (points, test, hint)

def check(name, *answer):
    """Grade an exercise: check("ex1", your_answer). Repeatable until correct."""
    if name not in _XP["checks"]:
        print(f"unknown exercise: {name!r} — available: {list(_XP['checks'])}")
        return
    points, test, hint = _XP["checks"][name]
    try:
        ok = bool(test(*answer))
        err = None
    except Exception as e:
        ok, err = False, f"{type(e).__name__}: {e}"
    if ok:
        first = name not in _XP["done"]
        if first:
            _XP["done"].add(name)
            _XP["earned"] += points
        total = sum(p for p, _, _ in _XP["checks"].values())
        tag = f"+{points} XP" if first else "already earned"
        print(f"✅ {name} — correct! {tag}   ⚡ {_XP['earned']}/{total} XP")
    else:
        msg = f"❌ {name} — not yet."
        if err:
            msg += f" (your answer raised {err})"
        print(msg + f"\n   💡 hint: {hint}")

def xp_report():
    total = sum(p for p, _, _ in _XP["checks"].values())
    n = len(_XP["checks"])
    bar = "█" * int(10 * _XP["earned"] / max(total, 1)) or "░"
    print(f"⚡ XP: {_XP['earned']}/{total}   [{bar:<10}]   exercises: {len(_XP['done'])}/{n}")
    for name in _XP["checks"]:
        print(("  ✅ " if name in _XP["done"] else "  ⬜ ") + name)

## Learning from consequences

Everything so far had ground-truth answers. In **reinforcement learning** an *agent* acts in an
*environment* and receives only *rewards* — no one ever says which action was correct. The agent
must discover a **policy** (state → action) that maximizes total reward.

Our arena: **CartPole** — push a cart left/right to keep a pole balanced. Reward: +1 per tick
alive. If `gymnasium` is installed we use it; otherwise a built-in physics clone kicks in.

In [ ]:
class MiniCartPole:
    """Physics-faithful CartPole clone so the quest never needs external deps."""
    def reset(self):
        self.s = (torch.rand(4) - 0.5) * 0.1
        self.n = 0
        return self.s.clone(), {}
    def step(self, a):
        x, xd, th, thd = self.s
        f = 10.0 if a == 1 else -10.0
        ct, st = math.cos(th), math.sin(th)
        tmp = (f + 0.05 * thd ** 2 * st) / 1.1
        thacc = (9.8 * st - ct * tmp) / (0.5 * (4 / 3 - 0.1 * ct ** 2 / 1.1))
        xacc = tmp - 0.05 * thacc * ct / 1.1
        dt = 0.02
        self.s = torch.tensor([x + dt * xd, xd + dt * xacc, th + dt * thd, thd + dt * thacc])
        self.n += 1
        done = bool(abs(self.s[0]) > 2.4 or abs(self.s[2]) > 0.21 or self.n >= 500)
        return self.s.clone(), 1.0, done, False, {}

try:
    import gymnasium as gym
    env = gym.make("CartPole-v1")
    print("using gymnasium CartPole-v1")
except Exception:
    env = MiniCartPole()
    print("gymnasium not found — using built-in MiniCartPole")

def as_t(obs):
    return obs if torch.is_tensor(obs) else torch.as_tensor(obs, dtype=torch.float32)

### Strategy A — REINFORCE: make good episodes more likely

The policy network outputs action *probabilities*. Play an episode, then nudge the policy so
that actions from **high-reward** episodes become more likely. The gradient of
`-log π(a) · return` does exactly that.

One subtlety: rewards should be **discounted** — an action deserves credit mostly for rewards
that follow *soon* after it.

In [ ]:
def discount(rewards, gamma=0.99):
    out, g = [], 0.0
    for r in reversed(rewards):
        g = r + gamma * g
        out.insert(0, g)
    out = torch.tensor(out)
    return (out - out.mean()) / (out.std() + 1e-8)     # normalize: huge variance reduction

policy = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 2))
opt = torch.optim.Adam(policy.parameters(), lr=1e-2)

R_hist = []
for ep in range(250):
    obs, _ = env.reset()
    logps, rewards = [], []
    done = False
    while not done:
        dist = torch.distributions.Categorical(logits=policy(as_t(obs)))
        a = dist.sample()
        logps.append(dist.log_prob(a))
        obs, r, done, trunc, _ = env.step(a.item())
        rewards.append(r); done = done or trunc
    loss = -(torch.stack(logps) * discount(rewards)).sum()
    opt.zero_grad(); loss.backward(); opt.step()
    R_hist.append(sum(rewards))
    if ep % 50 == 0:
        print(f"episode {ep:3d}: reward {R_hist[-1]:.0f}  (avg last 25: {sum(R_hist[-25:]) / len(R_hist[-25:]):.0f})")

plt.plot(R_hist, alpha=0.4)
plt.plot(torch.tensor(R_hist).unfold(0, 20, 1).mean(1), lw=2)
plt.title("REINFORCE learning to balance"); plt.xlabel("episode"); plt.ylabel("reward"); plt.show()

### Strategy B — DQN: learn the *value* of actions

Instead of a policy, learn **Q(state, action)** = expected future reward — then act greedily.
Two stabilizers made this work on Atari in 2015:
- a **replay buffer** (learn from shuffled past experience, not just the last step),
- a frozen **target network** (chase a stable target, not your own tail).

In [ ]:
from collections import deque

qnet = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 2))
target = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 2))
target.load_state_dict(qnet.state_dict())
opt = torch.optim.Adam(qnet.parameters(), lr=1e-3)
buf = deque(maxlen=8000)
eps, GAMMA = 1.0, 0.99

D_hist = []
for ep in range(160):
    obs, _ = env.reset()
    done, total = False, 0.0
    while not done:
        if random.random() < eps:
            a = random.randrange(2)
        else:
            with torch.no_grad():
                a = qnet(as_t(obs)).argmax().item()
        nxt, r, done, trunc, _ = env.step(a); done = done or trunc
        buf.append((as_t(obs), a, r, as_t(nxt), float(done)))
        obs, total = nxt, total + r

        if len(buf) >= 256:
            batch = random.sample(buf, 128)
            s, a_, r_, s2, d = map(lambda z: torch.stack(z) if torch.is_tensor(z[0]) else torch.tensor(z), zip(*batch))
            q = qnet(s).gather(1, a_.long().unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                tgt = r_ + GAMMA * target(s2).max(1).values * (1 - d)
            opt.zero_grad(); F.smooth_l1_loss(q, tgt).backward(); opt.step()

    eps = max(0.05, eps * 0.96)
    if ep % 15 == 0:
        target.load_state_dict(qnet.state_dict())
    D_hist.append(total)
    if ep % 40 == 0:
        print(f"episode {ep:3d}: reward {total:.0f}  epsilon {eps:.2f}")

plt.plot(D_hist, alpha=0.4)
plt.plot(torch.tensor(D_hist).unfold(0, 15, 1).mean(1), lw=2)
plt.title("DQN learning CartPole"); plt.xlabel("episode"); plt.ylabel("reward"); plt.show()

Two philosophies, one result: an agent that balances a pole having never been told how.
**Policy-based** (REINFORCE → PPO → RLHF, which aligns chatbots) and **value-based**
(DQN → Rainbow) are the two great families of RL.

In [ ]:
# ── this notebook's exercises (run once) ───────────────────────────────
_register("warmup", 5,
    lambda s: "reward" in s.lower(),
    "the only feedback an RL agent ever receives")
_register("discount_fn", 20,
    lambda f: (lambda o: len(o) == 3 and abs(o[0] - 2.9701) < 1e-3 and abs(o[1] - 1.99) < 1e-3 and abs(o[2] - 1.0) < 1e-9)(f([1.0, 1.0, 1.0], 0.99)),
    "raw discounted returns, NO normalization: g_t = r_t + gamma * g_{t+1}. discount([1,1,1], .99) -> [2.9701, 1.99, 1.0]")
_register("greedy_eps", 15,
    lambda f: 0.25 < sum(f([0.0, 5.0, 1.0], 0.9) != 1 for _ in range(500)) / 500 < 0.75,
    "def act(q, eps): return random action with prob eps, else argmax(q). Test uses eps=0.9")
_register("why_target", 10,
    lambda s: "stab" in s.lower() or "moving" in s.lower() or "chas" in s.lower() or "fixed" in s.lower(),
    "one phrase: why freeze a separate target network in DQN?")

In [ ]:
check("warmup", "reward")

## 🏆 Boss Challenges

Earn your XP — write each answer in a cell below and grade it with `check(...)`. When you're done, run `xp_report()`.

- `discount_fn` (20 XP) — write raw (unnormalized) discounted returns; submit the function.
- `greedy_eps` (15 XP) — write `act(q_values, eps)` doing epsilon-greedy selection; submit the function.
- `why_target` (10 XP) — why does DQN need a target network?

In [ ]:
# ⚔️ your attempts here...

# xp_report()